In [89]:
# Cellule 1: importe les bibliotheques et configure Altair.
import altair as alt
import pandas as pd
import geopandas as gpd # Requires geopandas -- e.g.: conda install -c conda-forge geopandas
alt.data_transformers.enable('default', max_rows=None) # Inline data so static rendering can access it
alt.renderers.enable('png') # Static renderer: avoids frontend vega-embed/requirejs issues in VS Code notebooks
import ipywidgets as widgets
from IPython.display import clear_output, display
from pathlib import Path

In [96]:
# Cellule 2: charge et prepare les donnees (nettoyage, agregats, rangs).
data_path = Path('..') / 'dpt2020.csv'
just_names = pd.read_csv(data_path, sep=';')
just_names = just_names[just_names['preusuel'] != '_PRENOMS_RARES']
just_names = just_names[just_names['dpt'] != 'XX']

years = sorted(just_names['annais'].astype(str).str.strip().astype(int).unique())

# Fixed color map for all first names (stable across years)
all_names = sorted(just_names['preusuel'].dropna().astype(str).unique())
base_colors = [
    '#4C78A8', '#F58518', '#E45756', '#72B7B2', '#54A24B',
    '#EECA3B', '#B279A2', '#FF9DA6', '#9D755D', '#BAB0AC',
    '#1F77B4', '#FF7F0E', '#2CA02C', '#D62728', '#9467BD',
    '#8C564B', '#E377C2', '#7F7F7F', '#BCBD22', '#17BECF',
    '#264653', '#2A9D8F', '#E9C46A', '#F4A261', '#E76F51',
    '#3A86FF', '#8338EC', '#FF006E', '#FB5607', '#06D6A0',
    '#118AB2', '#073B4C', '#8E9AAF', '#CBC0D3', '#EFD3D7'
]
color_range = [base_colors[i % len(base_colors)] for i in range(len(all_names))]

# Fixed x-axis scale for all years: global max + 5000
max_nombre_global = (
    just_names
    .groupby(['annais', 'preusuel'], as_index=False)['nombre']
    .sum()['nombre']
    .max()
)
x_max = int(max_nombre_global) + 5000

# Precompute shared aggregates once
yearly_name_totals = (
    just_names.assign(annais_clean=just_names['annais'].astype(str).str.strip().astype(int))
    .groupby(['annais_clean', 'preusuel'], as_index=False)['nombre']
    .sum()
)
annual_stats = (
    yearly_name_totals
    .groupby('annais_clean', as_index=False)
    .agg(
        births_this_year=('nombre', 'sum'),
        names_over_1000=('nombre', lambda series: int((series > 1000).sum()))
    )
    .set_index('annais_clean')
)

# Precompute yearly full ranking once to measure rank movement
ranks_by_year = {}
for year in years:
    year_ranking = (
        yearly_name_totals[yearly_name_totals['annais_clean'] == year]
        .sort_values(['nombre', 'preusuel'], ascending=[False, True])
        .reset_index(drop=True)
    )
    year_ranking['rank'] = year_ranking.index + 1
    ranks_by_year[year] = dict(zip(year_ranking['preusuel'], year_ranking['rank']))

def trend_label(delta):
    if pd.isna(delta) or int(delta) == 0:
        return ''
    delta_int = int(delta)
    return f'↑ {delta_int}' if delta_int > 0 else f'↓ {abs(delta_int)}'

def trend_color(delta):
    if pd.isna(delta) or int(delta) == 0:
        return '#6b7280'
    return '#16a34a' if int(delta) > 0 else '#dc2626'

In [97]:
# Cellule 3: cree les widgets de controle (slider annee, vitesse, play).
annee_slider = widgets.IntSlider(
    value=1900,
    min=years[0],
    max=years[-1],
    step=1,
    description='Année',
    continuous_update=False,
    readout=True,
    readout_format='d',
    tooltip='Choisir l année affichée'
)

vitesse_slider = widgets.FloatSlider(
    value=2.0,
    min=0.5,
    max=5.0,
    step=0.1,
    description='Durée (s)',
    continuous_update=False,
    readout=True,
    readout_format='.1f',
    tooltip='Durée en secondes par année (plus grand = plus lent)'
)

btn_play = widgets.ToggleButton(
    value=False,
    description='',
    icon='play',
    tooltip='Lancer le défilement des années',
    layout=widgets.Layout(width='44px')
)


play_engine = widgets.Play(
    value=annee_slider.value,
    min=annee_slider.min,
    max=annee_slider.max,
    step=1,
    interval=int(vitesse_slider.value * 1000),
    description=''
)

# Keep Play rendered (tiny/invisible) so ticks are reliable in VS Code.
play_engine.layout = widgets.Layout(width='1px', height='1px', opacity='0', overflow='hidden')
widgets.jslink((play_engine, 'value'), (annee_slider, 'value'))

top_card_widget = widgets.HTML()
stats_left_widget = widgets.HTML()
stats_right_widget = widgets.HTML()
chart_out = widgets.Output()

In [107]:
# Cellule 4: construit les fonctions de rendu et le cache lazy des cartes/graphes.
def stat_html(label, value):
    return f'''<div style="width:340px;height:60px;background:#e5e7eb;border-radius:6px;display:flex;flex-direction:column;justify-content:center;align-items:center;font-family:sans-serif;color:#1f2937;">
<div style="font-size:10px;color:#4b5563;line-height:1.05;">{label}</div>
<div style="font-size:16px;font-weight:700;line-height:1.05;">{value}</div>
</div>'''

def build_chart(top10_annee, annee):
    if annee == years[0]:
        top10_annee = top10_annee.copy()
        top10_annee['value_trend'] = ''
    name_order = top10_annee['preusuel'].tolist()

    bars_mark = alt.Chart(top10_annee).mark_bar().encode(
        x=alt.X(
            'nombre:Q',
            title='Nombre de naissances',
            axis=alt.Axis(domain=False),
            scale=alt.Scale(domain=[0, x_max])
        ),
        y=alt.Y(
            'preusuel:N',
            sort=name_order,
            title=None,
            axis=alt.Axis(domain=False, labels=False, ticks=False)
        ),
        color=alt.Color(
            'preusuel:N',
            legend=None,
            scale=alt.Scale(domain=all_names, range=color_range)
        ),
        tooltip=['preusuel:N', 'nombre:Q', 'prev_nombre:Q', 'nombre_delta:Q', 'rank:Q', 'prev_rank:Q', 'rank_delta:Q']
    )

    values_mark = alt.Chart(top10_annee).mark_text(
        align='left',
        baseline='middle',
        dx=5
    ).encode(
        x='nombre:Q',
        y=alt.Y(
            'preusuel:N',
            sort=name_order,
            axis=alt.Axis(domain=False)
        ),
        text=alt.Text('nombre:Q', format=',d')
    )

    values_trend_mark = alt.Chart(top10_annee[top10_annee['value_trend'] != '']).mark_text(
        align='left',
        baseline='middle',
        fontWeight='bold',
        dx=42
    ).encode(
        x='nombre:Q',
        y=alt.Y(
            'preusuel:N',
            sort=name_order,
            axis=alt.Axis(domain=False)
        ),
        text='value_trend:N',
        color=alt.Color('value_trend_color:N', scale=None, legend=None)
    )

    names_mark = alt.Chart(top10_annee).mark_text(
        align='right',
        baseline='middle',
        dx=-48,
        color='#1f2937'
    ).encode(
        x=alt.value(0),
        y=alt.Y(
            'preusuel:N',
            sort=name_order,
            axis=alt.Axis(domain=False)
        ),
        text='preusuel:N'
    )

    trend_mark = alt.Chart(top10_annee[top10_annee['trend'] != '']).mark_text(
        align='right',
        baseline='middle',
        fontWeight='bold',
        dx=-18
    ).encode(
        x=alt.value(0),
        y=alt.Y(
            'preusuel:N',
            sort=name_order,
            axis=alt.Axis(domain=False)
        ),
        text='trend:N',
        color=alt.Color('trend_color:N', scale=None, legend=None)
    )

    bars = bars_mark + values_mark + values_trend_mark + names_mark + trend_mark
    return bars.properties(
        title=f'Top 10 des prénoms en {annee}',
        width=700,
        height=350
    )

# Lazy caches: keep existing cache on rerun to avoid rebuilding.
if 'yearly_payload' not in globals():
    yearly_payload = {}
if 'cards_by_year' not in globals():
    cards_by_year = {}
if 'charts_by_year' not in globals():
    charts_by_year = {}

def get_year_payload(annee):
    if annee in yearly_payload:
        cached_payload = yearly_payload[annee]
        required_cols = {'prev_nombre', 'nombre_delta', 'value_trend', 'value_trend_color'}
        if required_cols.issubset(set(cached_payload['top10'].columns)):
            return cached_payload
        # Invalidate stale cache entries created before value-trend fields existed.
        yearly_payload.pop(annee, None)
        cards_by_year.pop(annee, None)
        charts_by_year.pop(annee, None)

    year_data = yearly_name_totals[yearly_name_totals['annais_clean'] == annee]
    top10_annee = (
        year_data
        .drop(columns=['annais_clean'])
        .sort_values('nombre', ascending=False)
        .head(10)
        .copy()
    )

    current_ranks = ranks_by_year[annee]
    previous_ranks = ranks_by_year.get(annee - 1, {})
    top10_annee['rank'] = top10_annee['preusuel'].map(current_ranks)
    top10_annee['prev_rank'] = top10_annee['preusuel'].map(previous_ranks)
    top10_annee['rank_delta'] = top10_annee['prev_rank'] - top10_annee['rank']
    top10_annee['trend'] = top10_annee['rank_delta'].apply(trend_label)
    top10_annee['trend_color'] = top10_annee['rank_delta'].apply(trend_color)

    previous_values = (
        yearly_name_totals[yearly_name_totals['annais_clean'] == annee - 1]
        .set_index('preusuel')['nombre']
        .to_dict()
    )
    top10_annee['prev_nombre'] = top10_annee['preusuel'].map(previous_values).fillna(0)
    top10_annee['nombre_delta'] = top10_annee['nombre'] - top10_annee['prev_nombre']

    def value_trend_label(delta):
        delta_int = int(delta)
        if delta_int > 0:
            return '↑'
        if delta_int < 0:
            return '↓'
        return ''

    def value_trend_color(delta):
        if int(delta) > 0:
            return '#16a34a'
        if int(delta) < 0:
            return '#dc2626'
        return '#6b7280'

    top10_annee['value_trend'] = top10_annee['nombre_delta'].apply(value_trend_label)
    top10_annee['value_trend_color'] = top10_annee['nombre_delta'].apply(value_trend_color)
    if annee == years[0]:
        top10_annee['value_trend'] = ''

    births_this_year = int(annual_stats.loc[annee, 'births_this_year'])
    names_over_1000 = int(annual_stats.loc[annee, 'names_over_1000'])
    top1_name = top10_annee.iloc[0]['preusuel'] if len(top10_annee) else ''

    payload = {
        'top10': top10_annee,
        'births_this_year': births_this_year,
        'names_over_1000': names_over_1000,
        'top1_name': top1_name
    }
    yearly_payload[annee] = payload
    return payload

def get_card_and_chart(annee):
    if annee in cards_by_year and annee in charts_by_year:
        return cards_by_year[annee], charts_by_year[annee]

    payload = get_year_payload(annee)
    top10_annee = payload['top10']
    births_this_year = payload['births_this_year']
    names_over_1000 = payload['names_over_1000']
    top1_name = payload['top1_name']

    top_card = f'''<div style="width:700px;height:80px;background:#ffd166;border-radius:0px;display:flex;justify-content:center;align-items:center;font-family:sans-serif;color:#1f2937;font-size:22px;font-weight:700;">
Prénom de l'année : {top1_name}
</div>'''
    stats_left = stat_html('Naissances cette année', f"{births_this_year:,}".replace(',', ' '))
    stats_right = stat_html('Prénoms > 1000 naissances', str(names_over_1000))

    cards_by_year[annee] = (top_card, stats_left, stats_right)
    charts_by_year[annee] = build_chart(top10_annee, annee)
    return cards_by_year[annee], charts_by_year[annee]

In [99]:
# Cellule 5: definit les callbacks et met en page le tableau de bord.
def render_all():
    annee = annee_slider.value
    (top_card, stats_left, stats_right), bars = get_card_and_chart(annee)

    top_card_widget.value = top_card
    stats_left_widget.value = stats_left
    stats_right_widget.value = stats_right

    with chart_out:
        clear_output(wait=True)
        display(bars)

def refresh_chart(_change=None):
    render_all()

def update_play_interval(_change):
    play_engine.interval = int(vitesse_slider.value * 1000)

def on_play_step(change):
    if change['name'] != 'value':
        return
    # Force year sync + redraw on each Play tick.
    new_year = int(change['new'])
    if annee_slider.value != new_year:
        annee_slider.value = new_year
    else:
        render_all()

def on_play_toggle(change):
    if change['name'] != 'value':
        return

    if change['new'] and annee_slider.value >= annee_slider.max:
        annee_slider.value = annee_slider.min
        play_engine.value = annee_slider.value

    btn_play.icon = 'pause' if change['new'] else 'play'
    btn_play.tooltip = (
        'Arrêter le défilement des années'
        if change['new']
        else 'Lancer le défilement des années'
    )

# Center cards and chart block on the same visual width.
stats_box = widgets.HBox(
    [stats_left_widget, stats_right_widget],
    layout=widgets.Layout(width='700px', justify_content='space-between', gap='16px')
)

# Add a small white margin around the chart.
chart_box = widgets.Box(
    [chart_out],
    layout=widgets.Layout(width='724px', padding='12px', margin='0px')
)


In [ ]:
# Cellule 6: relie les widgets, reinitialise l'etat et affiche l'interface.
# Cette cellule est autonome pour redemarrer la visualisation.
annee_slider = widgets.IntSlider(
    value=1900,
    min=years[0],
    max=years[-1],
    step=1,
    description='Année',
    continuous_update=False,
    readout=True,
    readout_format='d',
    tooltip='Choisir l année affichée'
)

vitesse_slider = widgets.FloatSlider(
    value=2.0,
    min=0.5,
    max=5.0,
    step=0.1,
    description='Durée (s)',
    continuous_update=False,
    readout=True,
    readout_format='.1f',
    tooltip='Durée en secondes par année (plus grand = plus lent)'
)

btn_play = widgets.ToggleButton(
    value=False,
    description='',
    icon='play',
    tooltip='Lancer le défilement des années',
    layout=widgets.Layout(width='44px')
)

play_engine = widgets.Play(
    value=annee_slider.value,
    min=annee_slider.min,
    max=annee_slider.max,
    step=1,
    interval=int(vitesse_slider.value * 1000),
    description=''
)

# Keep Play rendered (tiny/invisible) so ticks are reliable in VS Code.
play_engine.layout = widgets.Layout(width='1px', height='1px', opacity='0', overflow='hidden')

top_card_widget = widgets.HTML()
stats_left_widget = widgets.HTML()
stats_right_widget = widgets.HTML()
chart_out = widgets.Output()

# Center cards and chart block on the same visual width.
stats_box = widgets.HBox(
    [stats_left_widget, stats_right_widget],
    layout=widgets.Layout(width='700px', justify_content='space-between', gap='16px')
)

# Add a small white margin around the chart.
chart_box = widgets.Box(
    [chart_out],
    layout=widgets.Layout(width='724px', padding='12px', margin='0px')
)

# Drop old links if they exist, then recreate clean links.
if 'play_btn_link' in globals() and play_btn_link is not None:
    play_btn_link.unlink()
if 'play_value_link' in globals() and play_value_link is not None:
    play_value_link.unlink()

play_btn_link = widgets.jslink((btn_play, 'value'), (play_engine, 'playing'))
play_value_link = widgets.jslink((play_engine, 'value'), (annee_slider, 'value'))

# Force fresh chart/card cache so new arrow logic is always applied.
if 'cards_by_year' in globals():
    cards_by_year.clear()
if 'charts_by_year' in globals():
    charts_by_year.clear()

# Wire callbacks.
annee_slider.observe(refresh_chart, names='value')
vitesse_slider.observe(update_play_interval, names='value')
btn_play.observe(on_play_toggle, names='value')

# Reset controls on launch.
btn_play.value = False
play_engine.playing = False
annee_slider.value = annee_slider.min
play_engine.value = annee_slider.value
update_play_interval(None)

render_all()
display(widgets.VBox([
    top_card_widget,
    stats_box,
    chart_box,
    widgets.HBox([btn_play, annee_slider, vitesse_slider]),
    play_engine
], layout=widgets.Layout(align_items='center')))
